# ResNet-50 ImageNet / ImageNet-C Accuracy Check

Sanity-check notebook for evaluating a timm vision model on clean ImageNet from HF cache and ImageNet-C from either Kaggle/local ReservoirTTA layout or the old mixed HF mirror.

For the real ReservoirTTA-style check, use `IMAGENET_C_SOURCE = "local_kaggle"` with `data/imagenet-c/<corruption>/<severity>/<class>/*`.

In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/home/kim/Project/Feature-Reliance')

In [10]:
import time
from collections import defaultdict

import torch
import timm
from timm.data import create_transform, resolve_model_data_config
from tqdm.auto import tqdm

from reservoir_sae.utils.hf_data import (
    build_reservoirtta_imagenet_c_dataset,
    discover_imagenet_c_domains,
    build_timm_label_mapping,
    filter_corruption_dataset,
    infer_columns,
    load_hf_split,
    make_loader,
    make_torch_loader,
    resolve_imagenet_c_root,
    select_class_balanced_subset,
)

torch.set_grad_enabled(False)

In [11]:
# Edit these for fast/full runs.
CACHE_DIR = "data/hf_cache"
OFFLINE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "resnet50"
BATCH_SIZE = 128
NUM_WORKERS = 4

IMAGENET_DATASET = "ILSVRC/imagenet-1k"
IMAGENET_SPLIT = "validation"
IMAGENET_MAX_SAMPLES = 1024  # Set to None for all 50k validation images.
IMAGENET_SHUFFLE_SEED = 0

# "local" reads the repo-root data/imagenet-c layout: corruption/severity/wnid/images.
# "hf_mixed" is only a fallback sanity check for an HF mirror with image/label columns.
IMAGENET_C_SOURCE = "local"
IMAGENET_C_ROOT = PROJECT_ROOT / "data" / "imagenet-c"
IMAGENET_C_MAX_DOMAINS = None    # None uses every local corruption/severity domain found.
IMAGENET_C_MAX_SAMPLES = 1024    # local: examples per domain. hf_mixed: total examples.
IMAGENET_C_CORRUPTIONS = None    # None discovers all local corruptions, including extra.tar domains.
IMAGENET_C_SEVERITIES = None     # None discovers available severities under each corruption.

HF_IMAGENET_C_DATASET = "ang9867/ImageNet-C"
HF_IMAGENET_C_SPLIT = "train"
HF_IMAGENET_C_SHUFFLE_SEED = 0

print("device:", DEVICE)

device: cuda


In [12]:
t0 = time.perf_counter()
print(f"[start] build {MODEL_NAME}")
model = timm.create_model(MODEL_NAME, pretrained=True).to(DEVICE).eval()
transform = create_transform(**resolve_model_data_config(model), is_training=False)
print(f"[done] build {MODEL_NAME} ({time.perf_counter() - t0:.1f}s)")

[start] build resnet50
[done] build resnet50 (0.4s)


In [13]:
def load_hf_eval_loader(dataset_name, split, max_samples=None, corruptions=None, severities=None, shuffle_seed=None):
    t0 = time.perf_counter()
    print(f"[start] load HF {dataset_name} split={split}")
    dataset = load_hf_split(dataset_name=dataset_name, split=split, cache_dir=CACHE_DIR, offline=OFFLINE)
    columns = infer_columns(dataset)
    label_mapping = build_timm_label_mapping(dataset, columns)
    dataset = filter_corruption_dataset(dataset, columns, corruptions, severities, None)
    if max_samples is not None:
        print(f"class-balanced HF subset seed: {shuffle_seed}")
        dataset = select_class_balanced_subset(
            dataset,
            columns,
            max_samples,
            seed=0 if shuffle_seed is None else shuffle_seed,
            label_mapping=label_mapping,
        )
    loader = make_loader(dataset, transform, columns, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, label_mapping=label_mapping)
    print(f"label mapping: {label_mapping.source}; identity={label_mapping.is_identity}")
    print(f"columns: {dataset.column_names}")
    print(f"rows: {len(dataset):,}; batches: {len(loader):,}")
    print(f"[done] load HF {dataset_name} ({time.perf_counter() - t0:.1f}s)")
    return loader, {"source": "hf", "rows": len(dataset), "columns": dataset.column_names, "label_mapping": label_mapping.source}


def load_local_imagenet_c_loader():
    t0 = time.perf_counter()
    root = resolve_imagenet_c_root(IMAGENET_C_ROOT)
    domains = discover_imagenet_c_domains(root, corruptions=IMAGENET_C_CORRUPTIONS, severities=IMAGENET_C_SEVERITIES)
    print(f"[start] load local ImageNet-C root={root}")
    print(f"available domains after filters: {len(domains)}")
    dataset, segments = build_reservoirtta_imagenet_c_dataset(
        transform=transform,
        root=root,
        corruptions=IMAGENET_C_CORRUPTIONS,
        severities=IMAGENET_C_SEVERITIES,
        examples_per_domain=IMAGENET_C_MAX_SAMPLES,
        seed=0,
        max_domains=IMAGENET_C_MAX_DOMAINS,
    )
    loader = make_torch_loader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    print(f"resolved root: {root}")
    print(f"domains: {len(segments)}; rows: {len(dataset):,}; batches: {len(loader):,}")
    for segment in segments[:8]:
        print(
            f"  domain={segment['domain_index']:02d} {segment['corruption']}/{segment['severity']} "
            f"samples={segment['num_samples']:,} classes={segment.get('num_classes', 'n/a')} "
            f"per_class={segment.get('selected_min_per_class', 'n/a')}-{segment.get('selected_max_per_class', 'n/a')} "
            f"label_mapping={segment.get('label_mapping', 'n/a')}"
        )
    if len(segments) > 8:
        print(f"  ... {len(segments) - 8} more domains")
    print(f"[done] load local ImageNet-C ({time.perf_counter() - t0:.1f}s)")
    return loader, {"source": "local", "root": str(root), "rows": len(dataset), "segments": segments}


def load_imagenet_c_loader():
    if IMAGENET_C_SOURCE in {"local", "local_kaggle"}:
        return load_local_imagenet_c_loader()
    if IMAGENET_C_SOURCE == "hf_mixed":
        return load_hf_eval_loader(
            HF_IMAGENET_C_DATASET,
            HF_IMAGENET_C_SPLIT,
            max_samples=IMAGENET_C_MAX_SAMPLES,
            shuffle_seed=HF_IMAGENET_C_SHUFFLE_SEED,
        )
    raise ValueError(f"Unknown IMAGENET_C_SOURCE={IMAGENET_C_SOURCE!r}")


def evaluate_accuracy(model, loader, name):
    t0 = time.perf_counter()
    correct = 0
    total = 0
    group_stats = defaultdict(lambda: [0, 0])

    print(f"[start] evaluate {name}")
    for images, labels, meta in tqdm(loader, desc=name, total=len(loader)):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        pred = model(images).argmax(dim=1)
        batch_correct = pred.eq(labels).detach().cpu()

        correct += int(batch_correct.sum())
        total += int(labels.numel())

        corruptions = meta.get("corruption", [""] * len(batch_correct))
        severities = meta.get("severity", [-1] * len(batch_correct))
        for ok, corruption, severity in zip(batch_correct.tolist(), corruptions, severities):
            if corruption or severity != -1:
                key = (str(corruption), int(severity))
                group_stats[key][0] += int(ok)
                group_stats[key][1] += 1

    acc = correct / max(total, 1)
    print(f"[done] evaluate {name} ({time.perf_counter() - t0:.1f}s)")
    print(f"{name}: acc={acc:.4f} correct={correct:,} total={total:,}")

    by_group = []
    for (corruption, severity), (group_correct, group_total) in sorted(group_stats.items()):
        by_group.append({"corruption": corruption, "severity": severity, "acc": group_correct / max(group_total, 1), "correct": group_correct, "total": group_total})
    return {"name": name, "acc": acc, "correct": correct, "total": total, "by_group": by_group}

In [14]:
imagenet_loader, imagenet_info = load_hf_eval_loader(
    IMAGENET_DATASET,
    IMAGENET_SPLIT,
    max_samples=IMAGENET_MAX_SAMPLES,
    shuffle_seed=IMAGENET_SHUFFLE_SEED,
)
imagenet_result = evaluate_accuracy(model, imagenet_loader, "ImageNet validation")
imagenet_result

[start] load HF ILSVRC/imagenet-1k split=validation


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

class-balanced HF subset seed: 0
label mapping: HF description order matches timm ImageNet-1K; identity=True
columns: ['image', 'label']
rows: 1,024; batches: 8
[done] load HF ILSVRC/imagenet-1k (2.0s)
[start] evaluate ImageNet validation


ImageNet validation:   0%|          | 0/8 [00:00<?, ?it/s]

[done] evaluate ImageNet validation (1.7s)
ImageNet validation: acc=0.8145 correct=834 total=1,024


{'name': 'ImageNet validation',
 'acc': 0.814453125,
 'correct': 834,
 'total': 1024,
 'by_group': []}

In [15]:
imagenet_c_loader, imagenet_c_info = load_imagenet_c_loader()
imagenet_c_result = evaluate_accuracy(model, imagenet_c_loader, f"ImageNet-C ({IMAGENET_C_SOURCE})")
imagenet_c_result

[start] load local ImageNet-C root=/home/kim/Project/Feature-Reliance/data/imagenet-c
available domains after filters: 95
resolved root: /home/kim/Project/Feature-Reliance/data/imagenet-c
domains: 95; rows: 97,280; batches: 760
  domain=00 gaussian_noise/5 samples=1,024 classes=1000 per_class=1-2 label_mapping=local ImageNet-C WNID folder -> timm ImageNet-1K index
  domain=01 gaussian_noise/4 samples=1,024 classes=1000 per_class=1-2 label_mapping=local ImageNet-C WNID folder -> timm ImageNet-1K index
  domain=02 gaussian_noise/3 samples=1,024 classes=1000 per_class=1-2 label_mapping=local ImageNet-C WNID folder -> timm ImageNet-1K index
  domain=03 gaussian_noise/2 samples=1,024 classes=1000 per_class=1-2 label_mapping=local ImageNet-C WNID folder -> timm ImageNet-1K index
  domain=04 gaussian_noise/1 samples=1,024 classes=1000 per_class=1-2 label_mapping=local ImageNet-C WNID folder -> timm ImageNet-1K index
  domain=05 shot_noise/5 samples=1,024 classes=1000 per_class=1-2 label_mappi

ImageNet-C (local):   0%|          | 0/760 [00:00<?, ?it/s]

[done] evaluate ImageNet-C (local) (94.0s)
ImageNet-C (local): acc=0.4871 correct=47,385 total=97,280


{'name': 'ImageNet-C (local)',
 'acc': 0.48709909539473684,
 'correct': 47385,
 'total': 97280,
 'by_group': [{'corruption': 'brightness',
   'severity': 1,
   'acc': 0.6923828125,
   'correct': 709,
   'total': 1024},
  {'corruption': 'brightness',
   'severity': 2,
   'acc': 0.7265625,
   'correct': 744,
   'total': 1024},
  {'corruption': 'brightness',
   'severity': 3,
   'acc': 0.69921875,
   'correct': 716,
   'total': 1024},
  {'corruption': 'brightness',
   'severity': 4,
   'acc': 0.6806640625,
   'correct': 697,
   'total': 1024},
  {'corruption': 'brightness',
   'severity': 5,
   'acc': 0.6533203125,
   'correct': 669,
   'total': 1024},
  {'corruption': 'contrast',
   'severity': 1,
   'acc': 0.6865234375,
   'correct': 703,
   'total': 1024},
  {'corruption': 'contrast',
   'severity': 2,
   'acc': 0.6875,
   'correct': 704,
   'total': 1024},
  {'corruption': 'contrast',
   'severity': 3,
   'acc': 0.625,
   'correct': 640,
   'total': 1024},
  {'corruption': 'contrast',

In [16]:
summary = {
    "model": MODEL_NAME,
    "device": DEVICE,
    "imagenet": imagenet_result,
    "imagenet_c": imagenet_c_result,
    "imagenet_info": imagenet_info,
    "imagenet_c_info": imagenet_c_info,
}

print(f"ImageNet acc:   {summary['imagenet']['acc']:.4f} ({summary['imagenet']['total']:,} samples)")
print(f"ImageNet-C acc: {summary['imagenet_c']['acc']:.4f} ({summary['imagenet_c']['total']:,} samples, source={IMAGENET_C_SOURCE})")
summary

ImageNet acc:   0.8145 (1,024 samples)
ImageNet-C acc: 0.4871 (97,280 samples, source=local)


{'model': 'resnet50',
 'device': 'cuda',
 'imagenet': {'name': 'ImageNet validation',
  'acc': 0.814453125,
  'correct': 834,
  'total': 1024,
  'by_group': []},
 'imagenet_c': {'name': 'ImageNet-C (local)',
  'acc': 0.48709909539473684,
  'correct': 47385,
  'total': 97280,
  'by_group': [{'corruption': 'brightness',
    'severity': 1,
    'acc': 0.6923828125,
    'correct': 709,
    'total': 1024},
   {'corruption': 'brightness',
    'severity': 2,
    'acc': 0.7265625,
    'correct': 744,
    'total': 1024},
   {'corruption': 'brightness',
    'severity': 3,
    'acc': 0.69921875,
    'correct': 716,
    'total': 1024},
   {'corruption': 'brightness',
    'severity': 4,
    'acc': 0.6806640625,
    'correct': 697,
    'total': 1024},
   {'corruption': 'brightness',
    'severity': 5,
    'acc': 0.6533203125,
    'correct': 669,
    'total': 1024},
   {'corruption': 'contrast',
    'severity': 1,
    'acc': 0.6865234375,
    'correct': 703,
    'total': 1024},
   {'corruption': 'cont